# Reinforcement Learning for Stock Market Portfolio Management — reproduction notebook

**Ojetokun Oluwafemi Akinwale (190805019)** · B.Sc. Computer Science, University of Lagos · Supervisor: Dr B. A. Sawyerr

A comparative evaluation of three continuous-action deep RL algorithms — **PG**, **PPO** and **DDPG** — on an
eight-asset S&P 500 portfolio under 10 bps two-sided transaction costs.

---

## What this notebook does

| Section | Step | Cost |
|---|---|---|
| 1 | Fetch the code, install dependencies, run the 156-test gate | ~3 min |
| 2 | Show the locked configuration | instant |
| 3 | Rebuild the price panel and observation tensors from the **committed** CSVs, check invariants | ~10 s |
| 4 | Run the four baselines (UBAH, UCRP, BestStock, Markowitz) on all three splits | ~10 s |
| 5 | *(optional)* Retrain PG, PPO and DDPG | see below |
| 6 | Assemble the run view, compute the significance tables, render the figure suite | ~30 s |
| 7 | Display every figure and results table | instant |
| 8 | Download everything as a zip | ~10 s |

## Choose a mode in the next cell

| `MODE` | Seeds | Optimiser updates / seed | Colab wall-clock | What it gives you |
|---|---|---|---|---|
| `"figures_only"` | — | — | **~4 min** | The thesis results exactly as committed. **Start here.** |
| `"smoke"` | 3 | 3,000 | ~25 min | The whole pipeline end-to-end on a reduced budget. Numbers will *not* match the thesis. |
| `"quick"` | 3 | 15,000 | ~2 h | A recognisable but under-trained replication. |
| `"full"` | 10 | 60,000 | **~12–18 h** | The real experiment. Exceeds a free Colab session — read §5 first. |

Everything is CPU-bound: the shared feature extractor is 783 parameters, so a GPU runtime buys nothing.
Leave the runtime on **CPU**.

---
# 1 · Setup

**Before the GitHub path will work**, the branch must actually carry `src/stats.py`, `scripts/07_stats.py`
and `results/phase5/` — the Phase-5 significance machinery and the assembled run view. If they are still
sitting uncommitted on the author's machine, the check at the end of the next cell will say so; set
`SOURCE = "upload"` and upload a zip of the working tree instead.

In [ ]:
# ============================== SETTINGS ==============================
MODE   = "figures_only"     # "figures_only" | "smoke" | "quick" | "full"
SOURCE = "github"           # "github" -> git clone   |   "upload" -> upload a zip of the repo

GITHUB_URL = "https://github.com/FemiOje/thesis-drl-portfolio.git"
BRANCH     = "impl"

RUN_TESTS  = True           # the 156-test gate; ~45 s, worth it
# ======================================================================

PRESETS = {
    # total_timesteps drives everything: PPO and DDPG both perform total/5 optimiser
    # updates, so PG's gradient_steps is set to match. src/config.py refuses to load a
    # configuration in which the three budgets differ.
    "figures_only": None,
    "smoke": dict(seeds=3,  total_timesteps=15_000,  eval_every_steps=1_500),
    "quick": dict(seeds=3,  total_timesteps=75_000,  eval_every_steps=3_750),
    "full":  dict(seeds=10, total_timesteps=300_000, eval_every_steps=7_500),
}
assert MODE in PRESETS, MODE
PRESET = PRESETS[MODE]
RUN_ID = "phase5" if MODE == "figures_only" else f"colab_{MODE}"

print(f"MODE={MODE}   RUN_ID={RUN_ID}")
if PRESET:
    print(f"  {PRESET['seeds']} seeds x {PRESET['total_timesteps'] // 5:,} "
          f"optimiser updates per algorithm")

In [ ]:
import os, pathlib, shutil, subprocess, sys

REPO = pathlib.Path("/content/thesis-drl-portfolio")

if SOURCE == "github":
    if REPO.exists():
        print(f"{REPO} already present; leaving it alone (delete it to re-clone)")
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, GITHUB_URL,
                        str(REPO)], check=True)

elif SOURCE == "upload":
    # Zip the repo on your machine first. From the project root, either
    #     git archive --format=zip -o repo.zip HEAD          (committed state only)
    # or, to include uncommitted work,
    #     powershell Compress-Archive -Path config,data,docs,scripts,src,tests,requirements.txt,results -DestinationPath repo.zip -Force
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    shutil.rmtree(REPO, ignore_errors=True)
    REPO.mkdir(parents=True)
    shutil.unpack_archive(name, REPO)
    if not (REPO / "config" / "base.yaml").exists():          # zip nested one level
        inner = [p for p in REPO.iterdir() if (p / "config" / "base.yaml").exists()]
        for p in inner[0].iterdir():
            shutil.move(str(p), REPO)
else:
    raise ValueError(SOURCE)

os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("\ncwd:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout or "(no git metadata)")

# What this notebook needs beyond the Phase-4 tree.
required = ["config/base.yaml", "config/universe.yaml", "data/raw/AAPL.csv",
            "data/raw/^IRX.csv", "src/stats.py", "scripts/07_stats.py",
            "scripts/06_figures.py", "results/phase5/PG/test.npy"]
missing = [f for f in required if not (REPO / f).exists()]
if missing:
    print("\n!! MISSING:", missing)
    print("   The significance tables (F10) and figures_only mode need these.")
    print("   Either push the branch that contains them, or re-run with SOURCE='upload'.")
else:
    print("all required files present")

In [ ]:
# Dependencies.
#
# requirements.txt pins numpy 1.24.3 / pandas 2.0.3 / torch 2.1.0, none of which publish
# cp312 wheels — they cannot install on Colab's interpreter. Below is the nearest set
# that can, holding the two versions that affect behaviour (gymnasium 0.29.1,
# stable-baselines3 2.2.1) at the thesis pins and keeping numpy on the 1.x series SB3
# 2.2.1 was written against. Colab's preinstalled torch is used as-is: the policies are
# 783-parameter CPU convolutions, so the torch minor version is immaterial here.
import sys
print("python", ".".join(map(str, sys.version_info[:2])))

if sys.version_info[:2] == (3, 10):
    print("exact thesis pins are installable on this interpreter")
    !pip install -q -r requirements.txt 2>&1 | tail -3
else:
    !pip install -q "numpy==1.26.4" "pandas>=2.0,<2.3" "scipy>=1.11,<1.14" "matplotlib>=3.7,<3.10" "seaborn==0.12.2" "gymnasium==0.29.1" "stable-baselines3==2.2.1" "PyYAML==6.0.1" "pytest==7.4.4" "yfinance==0.2.54" 2>&1 | tail -3
print("done")

In [ ]:
# Version report and the test gate. If this cell is green the environment is sound.
import importlib, subprocess, sys

for m in ("numpy", "pandas", "scipy", "torch", "gymnasium", "stable_baselines3",
          "matplotlib", "seaborn", "yaml"):
    try:
        print(f"  {m:20s} {importlib.import_module(m).__version__}")
    except Exception as e:
        print(f"  {m:20s} !! {e}")

if RUN_TESTS:
    print("\nrunning the gate ...")
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--no-header"],
                       capture_output=True, text=True)
    print("\n".join(r.stdout.strip().splitlines()[-12:]))
    if r.returncode:
        print("\n!! tests failed — nothing below is trustworthy until this is green")

---
# 2 · The locked configuration

Nothing in this codebase reads a magic number: every parameter arrives from `config/base.yaml` through
`src/config.py`, which **validates it at load time**. Three failure modes here are silent — they produce
plausible results from a broken setup — so all three are caught before anything runs:

- `tau` too low structurally caps the maximum single-asset weight (at `tau = 1` no asset can exceed ~48%);
- `gamma != 1.0` confounds the learning rule with the planning horizon (SB3 defaults to 0.99);
- unequal training budgets confound *which algorithm* with *which one got more updates*.

In [ ]:
if PRESET:
    # Rewrite the budget in config/base.yaml rather than passing command-line flags: the
    # config file is the single source of truth, and load_config() then re-checks that
    # all three algorithms still receive an identical number of optimiser updates.
    import re, pathlib
    T, ev = PRESET["total_timesteps"], PRESET["eval_every_steps"]
    grad = T // 5                      # PPO: (T/500)*10*10   DDPG: (T/500)*100
    p = pathlib.Path("config/base.yaml")
    txt = p.read_text(encoding="utf-8")
    seeds = ", ".join(str(i) for i in range(PRESET["seeds"]))
    txt = re.sub(r"^  seeds: \[.*?\]", f"  seeds: [{seeds}]", txt, count=1, flags=re.M)
    txt = re.sub(r"^    gradient_steps: 60000", f"    gradient_steps: {grad}",
                 txt, count=1, flags=re.M)
    # PG counts in gradient steps, the SB3 arms in env steps; ev // 5 is the same
    # interval on both clocks, so F1/F2/F4 keep a shared x-axis.
    txt = re.sub(r"^    eval_every: 1500", f"    eval_every: {max(1, ev // 5)}",
                 txt, count=1, flags=re.M)
    txt = re.sub(r"^    total_timesteps: 300000", f"    total_timesteps: {T}", txt, flags=re.M)
    txt = re.sub(r"^    eval_every_steps: 7500", f"    eval_every_steps: {ev}", txt, flags=re.M)
    p.write_text(txt, encoding="utf-8")
    print(f"budget rewritten: {grad:,} optimiser updates per algorithm, "
          f"seeds 0..{PRESET['seeds'] - 1}\n")

from src.config import load_config, gradient_steps, batch_sizes
import src.universe as U

cfg = load_config()          # raises ConfigError if any of the above is inconsistent

print(f"universe      {U.HEADLINE}")
print(f"window        {cfg.env.window} d x {cfg.env.n_features} features "
      f"-> observation {cfg.env.tensor_shape(cfg.universe.n_assets)}")
print(f"tau           {cfg.env.tau}   (max reachable single-asset weight "
      f"{cfg.env.max_reachable_weight(cfg.universe.n_assets):.3f})")
print(f"commission    {cfg.env.commission} both sides, "
      f"{cfg.env.mu_iterations} fixed-point iterations")
print(f"gamma         {cfg.agent.gamma}  (all three agents)")
print(f"seeds         {list(cfg.run_seeds)}")
print(f"budget        {gradient_steps(cfg.agent)}   batch {batch_sizes(cfg.agent)}")
for s in cfg.data.splits:
    print(f"split {s.name:9s} {s.start} .. {s.end}")

---
# 3 · Data

The raw per-ticker CSVs are **committed to the repository**. That is deliberate: Yahoo's endpoints change,
and reproducibility must not depend on a live API. The cell below therefore reads from disk and downloads
nothing. It resolves every universe, applies the pre-registered 0.70 correlation gate on the *train split
only*, builds the Eq. 18 observation tensors and asserts a set of invariants — including that the lookback
window is causal, checked against a direct recompute.

The eight names are not hand-picked. Sectors are fixed on economic grounds, and within each sector the
largest S&P 500 constituent by market capitalisation at 2021-08-25 is taken mechanically.

In [ ]:
!python scripts/01_build_data.py

In [ ]:
import pathlib
from IPython.display import Image, Markdown, display

P1 = pathlib.Path("results/phase1/figures")
for name, cap in [
    ("F0a_prices",
     "**F0a** — adjusted closes for the eight-asset universe, with the 60/20/20 chronological split boundaries."),
    ("F0b_correlation",
     "**F0b** — pairwise correlation of simple daily returns, measured on the train split only. The gate is 0.70 and the worst pair (AAPL/AMZN) reaches 0.576, so no substitution was triggered."),
    ("F0c_tensor",
     "**F0c** — the Eq. 18 observation tensor at one decision date: close, high and low over a 20-day window, each divided by the latest close."),
]:
    f = P1 / f"{name}.png"
    if f.exists():
        display(Markdown(cap)); display(Image(str(f)))

---
# 4 · Baselines

Four non-learning strategies, run through the *same* environment and the *same* cost model as the agents:

- **UBAH** — uniform buy-and-hold; bought once, never rebalanced.
- **UCRP** — uniform constant-rebalanced portfolio; rebalanced to 1/9 daily, so it pays costs.
- **BestStock** — the single best asset *chosen with hindsight*. An upper reference, not a strategy, and
  drawn dashed everywhere for that reason.
- **Markowitz** — mean-variance, re-estimated on a rolling window.

In [ ]:
!python scripts/02_run_baselines.py

---
# 5 · Training  *(skipped when `MODE = "figures_only"`)*

The comparison claims exactly one free axis: **the learning rule**. Everything else is held identical and
asserted in `tests/test_sb3.py` — the 783-parameter `EIIEExtractor` (parameter-for-parameter, including both
of DDPG's target networks), the observation, the action space, the `softmax(tau·a)` projection, `gamma`, the
cost model, the minibatch size, the checkpoint rule and the evaluation protocol. `docs/ARCHITECTURE_TABLE.md`
carries the full ledger, including the asymmetries that are *inherent* to the algorithms and so are reported
rather than hidden: DDPG's critic capacity, SB3's unavoidable `action_net`, and the fact that PG collects no
environment steps at all.

Budgets are equalised on the axis that was in our gift — **60,000 optimiser updates of batch 50** — and the
realised count is asserted against the prediction at the end of every run, because a prediction is not
evidence.

### `"full"` mode and Colab session limits

Ten seeds of all three algorithms is roughly **12–18 hours** of CPU. A free Colab session will be reclaimed
long before that. Three ways through it:

1. **One algorithm per session.** `scripts/05_train_sb3.py` writes each seed's artefacts the moment that seed
   finishes and skips any seed already on disk, so PPO and DDPG resume cleanly. Save `results/` to Drive
   (§8) between sessions and restore it before the training cells in the next one.
2. **Colab Pro with background execution**, which survives a closed tab.
3. **Use `"quick"`** for the mechanism, and read the committed `results/phase5` for the headline numbers.

`scripts/03_train_pg.py` has no resume path — but PG is also the cheapest arm, roughly 8 minutes per seed at
full budget.

In [ ]:
import time
if MODE == "figures_only":
    print("figures_only: skipping training, reading the committed results/{pg,ppo,ddpg}")
else:
    t0 = time.time()
    !python scripts/03_train_pg.py --run-id {RUN_ID}_pg
    print(f"\nPG total: {(time.time() - t0) / 60:.1f} min")

In [ ]:
import time
if MODE == "figures_only":
    print("skipped")
else:
    t0 = time.time()
    !python scripts/05_train_sb3.py --algo ppo --run-id {RUN_ID}_ppo
    print(f"\nPPO total: {(time.time() - t0) / 60:.1f} min")

In [ ]:
import time
if MODE == "figures_only":
    print("skipped")
else:
    t0 = time.time()
    !python scripts/05_train_sb3.py --algo ddpg --run-id {RUN_ID}_ddpg
    print(f"\nDDPG total: {(time.time() - t0) / 60:.1f} min")

---
# 6 · Assemble, test for significance, render

`results/<RUN_ID>/` is a *view*: per-seed wealth curves and training histories copied verbatim from the three
per-algorithm run directories, plus the concatenated metrics. The figure and statistics scripts read the
view, so they are indifferent to how it was produced. `results/phase5` is exactly this, built from the
committed `results/{pg,ppo,ddpg}`.

In [ ]:
import json, pathlib, shutil
import pandas as pd

def assemble(run_id, sources):
    """results/{pg,ppo,ddpg} -> one view at results/<run_id>. Curves and histories are
    copied verbatim; metrics are concatenated."""
    out = pathlib.Path("results") / run_id
    out.mkdir(parents=True, exist_ok=True)
    meta = {"note": "view assembled from " + ", ".join(sources.values()), "sources": {}}
    for algo, src in sources.items():
        src = pathlib.Path(src)
        (out / algo).mkdir(exist_ok=True)
        for f in ("train.npy", "validate.npy", "test.npy", "history.npz"):
            if (src / algo / f).exists():
                shutil.copy2(src / algo / f, out / algo / f)
        m = json.loads((src / "meta.json").read_text())
        meta["sources"][algo.lower()] = {"seeds": m.get("seeds"),
                                         "realised_updates": m.get("realised_updates")}
    for csv in ("metrics.csv", "metrics_per_seed.csv"):
        pd.concat([pd.read_csv(pathlib.Path(s) / csv) for s in sources.values()],
                  ignore_index=True).to_csv(out / csv, index=False, float_format="%.6f")
    (out / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    return out

if MODE == "figures_only":
    print("using the committed results/phase5")
else:
    print("->", assemble(RUN_ID, {"PG":   f"results/{RUN_ID}_pg",
                                  "PPO":  f"results/{RUN_ID}_ppo",
                                  "DDPG": f"results/{RUN_ID}_ddpg"}))

meta = json.loads(pathlib.Path(f"results/{RUN_ID}/meta.json").read_text())
print(json.dumps(meta["sources"], indent=2))

In [ ]:
# Paired t-tests of each agent against each baseline on daily log-returns, Bonferroni
# over the 3 x 4 family, plus bootstrap CIs on the mean daily return difference.
# The test split is scored ONCE; this reads committed curves and does not re-touch it.
!python scripts/07_stats.py --run-id {RUN_ID} --split test
print("\n" + "=" * 78 + "\n")
!python scripts/07_stats.py --run-id {RUN_ID} --split validate

In [ ]:
!python scripts/06_figures.py {RUN_ID}

---
# 7 · Results

In [ ]:
import pathlib
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.width", 220, "display.max_columns", 40)

RUN = pathlib.Path("results") / RUN_ID
allm = pd.concat([pd.read_csv("results/phase3/baselines.csv"),
                  pd.read_csv(RUN / "metrics.csv")], ignore_index=True)

ORDER = ["PG", "PPO", "DDPG", "UCRP", "UBAH", "Markowitz", "BestStock"]
COLS = ["final_value", "CR", "AR", "sharpe", "sortino", "MDD", "turnover",
        "max_weight", "HHI", "win_rate"]

for split in ("test", "validate", "train"):
    d = allm[allm.split == split].set_index("strategy").reindex(ORDER).dropna(how="all")[COLS]
    display(Markdown(f"### {split} split &nbsp;·&nbsp; agent rows are the **median across seeds**"))
    display(d.style.format("{:.4f}")
             .background_gradient(subset=["sharpe", "final_value"], cmap="Blues"))

In [ ]:
# Per-seed dispersion. The median row above hides how wide it is.
ps = pd.read_csv(RUN / "metrics_per_seed.csv")
for split in ("test", "validate"):
    d = ps[ps.split == split]
    display(Markdown(f"### per-seed final wealth and Sharpe — {split}"))
    display(d.pivot_table(index="seed", columns="strategy",
                          values=["final_value", "sharpe"]).round(4))
    display(d.groupby("strategy")[["final_value", "sharpe", "MDD", "turnover", "max_weight"]]
             .agg(["min", "median", "max"]).round(4))

In [ ]:
# Significance. level == "seeds" is the across-seed test; n_better counts how many of the
# run's seeds beat that baseline, and reject applies the Bonferroni-adjusted alpha.
for split in ("test", "validate"):
    f = RUN / f"stats_{split}.csv"
    if not f.exists():
        continue
    sl = pd.read_csv(f).query("level == 'seeds'").copy()
    sl["bp/day"] = sl.ret_diff * 1e4
    sl["bp_lo"] = sl.ret_lo * 1e4
    sl["bp_hi"] = sl.ret_hi * 1e4
    display(Markdown(f"### agents vs baselines — {split} &nbsp;·&nbsp; "
                     f"alpha_adj = {sl.alpha_adj.iloc[0]:.5f} over m = {int(sl.m.iloc[0])} comparisons"))
    display(sl[["algo", "baseline", "n_better", "mean_diff", "p", "p_adj", "reject",
                "bp/day", "bp_lo", "bp_hi"]].round(5).reset_index(drop=True))

## Figures

In [ ]:
import pathlib
from IPython.display import Image, Markdown, display

FIGS = pathlib.Path("results/figures")
CAPTIONS = [
    ("F1_learning_curves",
     "**F1 — learning curves.** Mean environment log-return against optimiser step, one band per algorithm across seeds. All three share a gradient-step x-axis by construction, which is the whole point of equalising the budget."),
    ("F2_train_val_vs_step",
     "**F2 — train against validation wealth, by step.** The gap is the story: with 735 training days the generalisation gap, not the optimiser, is the binding constraint."),
    ("F3_loss",
     "**F3 — the training objective (Eq. 21).**"),
    ("F4_plateau",
     "**F4 — where each run stops improving.**"),
    ("F18_training_convergence",
     "**F18 — convergence against the UCRP and BestStock references.**"),
    ("F5_test_wealth",
     "**F5 — test-split wealth.** Agents against the baselines they have to beat. Bands are the across-seed IQR; baselines have a single seed, so their bands collapse. BestStock is dashed because it is a hindsight upper reference, not a strategy."),
    ("F6_train_val_wealth",
     "**F6 — train and validation wealth.**"),
    ("F7_drawdown",
     "**F7 — test wealth with the drawdown panel.**"),
    ("F8_metrics_test",
     "**F8 — normalised metric heatmap, test split.** MDD and turnover are inverted, so brighter is always better."),
    ("F8_metrics_validate",
     "**F8 — normalised metric heatmap, validation split.**"),
    ("F8_metrics_train",
     "**F8 — normalised metric heatmap, train split.**"),
    ("F9_seed_distributions_test",
     "**F9 — per-seed dispersion, test split**, against the baseline levels."),
    ("F9_seed_distributions_validate",
     "**F9 — per-seed dispersion, validation split.**"),
    ("F10_forest_test",
     "**F10 — paired differences against each baseline, test split**, with Bonferroni-adjusted bootstrap CIs."),
    ("F10_forest_validate",
     "**F10 — the same on validation.** The validate-against-test pair is the overfitting evidence: apparent significance on the split used for checkpoint selection, none on the split scored once."),
]

shown = 0
for name, cap in CAPTIONS:
    f = FIGS / f"{name}.png"
    if f.exists():
        display(Markdown(cap)); display(Image(str(f))); shown += 1
print(f"{shown}/{len(CAPTIONS)} figures displayed")
missing = [n for n, _ in CAPTIONS if not (FIGS / f"{n}.png").exists()]
if missing:
    print("not produced by this run:", missing)

---
# 8 · Take the results with you

In [ ]:
import pathlib, shutil
shutil.make_archive("/content/drl_portfolio_results", "zip", root_dir="results")
z = pathlib.Path("/content/drl_portfolio_results.zip")
print(f"{z}  ({z.stat().st_size / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(str(z))
except Exception as e:
    print("(not running on Colab; the zip is on disk)", e)

In [ ]:
# Optional: persist results/ to Drive so a "full" run can be resumed in a later session.
SAVE_TO_DRIVE = False
DRIVE_DIR = "/content/drive/MyDrive/thesis-drl-portfolio"

if SAVE_TO_DRIVE:
    import pathlib, shutil
    from google.colab import drive
    drive.mount("/content/drive")
    dst = pathlib.Path(DRIVE_DIR) / "results"
    shutil.copytree("results", dst, dirs_exist_ok=True)
    print("->", dst)
    # To resume in the next session, run this BEFORE the training cells:
    #     shutil.copytree(dst, "results", dirs_exist_ok=True)

---

## Reading the output

The headline claim is an *ordering*, and the honest version of it lives in the gap between F9/F10 on
validation and F9/F10 on test. Checkpoints are selected by argmax validation final wealth, so validation is
not a clean split — apparent significance there is partly selection. The test split is scored once.

With ten seeds, a 251-day test window and a Bonferroni correction over twelve comparisons, this design has
limited power to separate three algorithms that all sit close to UCRP. Read `n_better` alongside `p_adj`:
"how many seeds beat this baseline" survives the small sample better than the p-value does.

## Provenance

- `docs/IMPLEMENTATION_PLAN.md` — the engineering contract: locked parameters, module contracts, phase gates.
- `docs/ARCHITECTURE_TABLE.md` — what was held identical, what differs, and which differences were chosen.
- `1706.10059.pdf` — Jiang, Xu & Liang (2017), the mathematical formalism. **Cite v1**; later versions renumber.
- `1808.09940v3.pdf` — Liang et al. (2018), the template being replicated.